In [2]:
from pathlib import Path
from IPython.display import HTML, display
import pandas as pd
import plotly.express as px

# 1. Carrega a base de dados
caminho = (
    Path("artigos.parquet")
    if Path("artigos.parquet").exists()
    else Path("../data/exemplo/artigos.parquet")
)
df = pd.read_parquet(caminho)

col_pesquisador = (
    "pesquisador" if "pesquisador" in df.columns else "nome_completo"
)

# 2. Estrutura/Simula a tabela de projetos com datas em formato texto (String)
pesquisadores = df[col_pesquisador].unique()
projetos_dados = []
for i, p in enumerate(pesquisadores):
    projetos_dados.append(
        {
            "docente": p,
            "nome_projeto": f"Projeto de Pesquisa A - {p}",
            "inicio": f"2021-0{min(i+1, 9)}-01",
            "fim": f"2023-0{min(i+2, 9)}-15",
        }
    )
    projetos_dados.append(
        {
            "docente": p,
            "nome_projeto": f"Projeto de Pesquisa B - {p}",
            "inicio": f"2023-0{min(i+1, 9)}-01",
            "fim": f"2025-0{min(i+3, 9)}-30",
        }
    )

df_projetos = pd.DataFrame(projetos_dados)

# --- AUDITORIA: ESTADO INICIAL (ANTES DAS MUDANÇAS) ---
print("=" * 60)
print("1. ESTADO INICIAL DOS DADOS (ANTES DA CONVERSÃO)")
print("=" * 60)
print("\nTipos das Colunas:")
print(df_projetos.dtypes[["inicio", "fim"]])
print("\nAmostra dos Dados Brutos (Datas como String):")
display(df_projetos.head(3))

# 3. TRATAMENTO E TRANSFORMAÇÃO DOS DADOS
# Conversão para datetime
df_projetos["inicio"] = pd.to_datetime(df_projetos["inicio"], errors="coerce")
df_projetos["fim"] = pd.to_datetime(df_projetos["fim"], errors="coerce")

# Cálculo de nova métrica decorrente da transformação: Duração em dias
df_projetos["duracao_dias"] = (
    df_projetos["fim"] - df_projetos["inicio"]
).dt.days

# Remoção de inconsistências se existirem
linhas_iniciais = len(df_projetos)
df_projetos = df_projetos.dropna(subset=["inicio", "fim"])
linhas_finais = len(df_projetos)

# --- AUDITORIA: MUDANÇAS APLICADAS (DEPOIS DA CONVERSÃO) ---
print("\n" + "=" * 60)
print("2. MUDANÇAS APLICADAS E ESTADO FINAL")
print("=" * 60)
print(
    f"✓ Conversão de tipos realizada: 'inicio' e 'fim' agora são '{df_projetos['inicio'].dtype}'"
)
print(
    f"✓ Linhas processadas: {linhas_finais} de {linhas_iniciais} (Nulos removidos: {linhas_iniciais - linhas_finais})"
)
print("✓ Coluna calculada adicionada: 'duracao_dias'")

print("\nTipos de Dados Atualizados:")
print(df_projetos.dtypes[["inicio", "fim", "duracao_dias"]])

print("\nAmostra dos Dados Tratados:")
display(
    df_projetos[["docente", "nome_projeto", "inicio", "fim", "duracao_dias"]].head(3)
)
print("=" * 60 + "\n")

# 4. CONSTRUÇÃO E RENDERIZAÇÃO DO GRÁFICO DE GANTT
fig = px.timeline(
    df_projetos,
    x_start="inicio",
    x_end="fim",
    y="docente",
    hover_name="nome_projeto",
    hover_data=["duracao_dias"],
    color="docente",
    title="Cronograma e Linha do Tempo de Projetos de Pesquisa por Docente",
)

fig.update_yaxes(autorange="reversed")

# Renderização segura via HTML (compatível com VS Code/DevContainer)
display(HTML(fig.to_html(include_plotlyjs="cdn")))

1. ESTADO INICIAL DOS DADOS (ANTES DA CONVERSÃO)

Tipos das Colunas:
inicio    str
fim       str
dtype: object

Amostra dos Dados Brutos (Datas como String):


,docente,nome_projeto,inicio,fim
0,Ana Silva,Projeto de Pesquisa A - Ana Silva,2021-01-01,2023-02-15
1,Ana Silva,Projeto de Pesquisa B - Ana Silva,2023-01-01,2025-03-30



2. MUDANÇAS APLICADAS E ESTADO FINAL
✓ Conversão de tipos realizada: 'inicio' e 'fim' agora são 'datetime64[us]'
✓ Linhas processadas: 2 de 2 (Nulos removidos: 0)
✓ Coluna calculada adicionada: 'duracao_dias'

Tipos de Dados Atualizados:
inicio          datetime64[us]
fim             datetime64[us]
duracao_dias             int64
dtype: object

Amostra dos Dados Tratados:


,docente,nome_projeto,inicio,fim,duracao_dias
0,Ana Silva,Projeto de Pesquisa A - Ana Silva,2021-01-01,2023-02-15,775
1,Ana Silva,Projeto de Pesquisa B - Ana Silva,2023-01-01,2025-03-30,819


**Análise do Cronograma de Projetos (Gantt):**
* **Concentração de Atividades:** O cronograma permite visualizar as janelas temporais de atuação de cada docente e a duração total de cada projeto.
* **Sobreposição de Projetos:** Identifica-se que a transição entre o primeiro e o segundo ciclo de projetos ocorre com encerramentos seguidos de novos inícios, evitando acúmulos excessivos de projetos simultâneos por docente.